# FFAI ResponseOptions API

This notebook introduces the `ResponseOptions` dataclass and the
`generate_response` signature introduced in the FFAI refactor.

Topics covered:

1. **New `generate_response` signature** -- `prompt`, `prompt_name`, `history`, and `options`
2. **`ResponseOptions` fields** -- `response_model`, `condition`, `response_format`, `strict`, and more
3. **Combining options** -- using multiple `ResponseOptions` fields together
4. **`history` as a top-level parameter** -- multi-turn conversation with history
5. **Provider kwargs** -- `temperature`, `max_tokens`, `tools` pass through as `**kwargs`

This notebook uses the **real Mistral API** via `FFMistralSmall`.

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from pydantic import BaseModel, Field  # noqa: E402

from src.Clients.FFMistralSmall import FFMistralSmall  # noqa: E402
from src.core.response_options import ResponseOptions  # noqa: E402
from src.FFAI import FFAI  # noqa: E402

client = FFMistralSmall(
    max_tokens=1024,
    temperature=0.3,
)

ffai = FFAI(client)

print(f"FFAI initialized with model: {client.model}")
print(f"ResponseOptions fields: {', '.join(ResponseOptions.__dataclass_fields__.keys())}")

---

## 1. Basic Call (no options)

The simplest call passes just a prompt string. No `ResponseOptions` needed.

In [ ]:
result = ffai.generate_response(
    "Explain quantum entanglement in one sentence.",
    prompt_name="basic",
)

print(f"Response: {result.response}")
print(f"Model: {result.model}")
print(f"Prompt name: {result.prompt_name}")

---

## 2. Structured Output with `response_model`

Pass a Pydantic model via `ResponseOptions(response_model=...)`. The LLM's
response is validated against the schema and returned as `result.parsed`.

In [ ]:
class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    rating: int = Field(ge=1, le=10, description="Rating out of 10")
    verdict: str = Field(description="One-word verdict: see, skip, or maybe")


result = ffai.generate_response(
    "Review the movie 'The Matrix'",
    prompt_name="movie_review",
    options=ResponseOptions(response_model=MovieReview),
)

print(f"Raw response:\n{result.response}\n")
print(f"Parsed: {result.parsed}")
print(f"Title: {result.parsed.title}")
print(f"Rating: {result.parsed.rating}/10")
print(f"Verdict: {result.parsed.verdict}")

---

## 3. Conditional Execution

Use `ResponseOptions(condition=...)` to skip the LLM call when a condition
evaluates to falsy. Useful for gating expensive API calls.

In [ ]:
result_skip = ffai.generate_response(
    "Translate to French: hello",
    prompt_name="translate_skip",
    options=ResponseOptions(condition=False),
)

print(f"Skipped: {result_skip.skipped}")
print(f"Response when skipped: {result_skip.response!r}")


result_run = ffai.generate_response(
    "Translate to French: hello",
    prompt_name="translate_run",
    options=ResponseOptions(condition=True),
)

print(f"\nSkipped: {result_run.skipped}")
print(f"Response: {result_run.response}")

---

## 4. JSON Mode with `response_format`

For raw JSON output without Pydantic validation, use
`ResponseOptions(response_format={'type': 'json_object'})`.

In [ ]:
import json

result = ffai.generate_response(
    "List 3 colors as a JSON object with keys 'colors' (array of strings).",
    prompt_name="json_colors",
    options=ResponseOptions(response_format={"type": "json_object"}),
)

print(f"Response:\n{result.response}")

parsed = json.loads(result.response)
print(f"\nParsed colors: {parsed['colors']}")

---

## 5. Multi-Turn with `history`

`history` is a top-level parameter on `generate_response`, not inside
`ResponseOptions`. Pass a list of message dicts to continue a conversation.

In [ ]:
history = []

result1 = ffai.generate_response(
    "My favorite color is blue. Remember that.",
    prompt_name="turn1",
    history=history,
)
history = result1.history
print(f"Turn 1: {result1.response}")

result2 = ffai.generate_response(
    "What is my favorite color?",
    prompt_name="turn2",
    history=history,
)
print(f"Turn 2: {result2.response}")

---

## 6. Combining Options

`ResponseOptions` fields can be combined. Here we use `condition` +
`response_model` together.

In [ ]:
class Translation(BaseModel):
    original: str
    translated: str
    language: str


result = ffai.generate_response(
    "Translate 'goodbye' to Spanish",
    prompt_name="translation",
    options=ResponseOptions(
        response_model=Translation,
        condition=True,
    ),
)

print(f"Parsed: {result.parsed}")
print(f"{result.parsed.original} -> {result.parsed.translated} ({result.parsed.language})")

---

## 7. Provider Kwargs

Parameters like `temperature`, `max_tokens`, and `tools` are passed directly
as `**kwargs` to the provider. They are NOT inside `ResponseOptions`.

In [ ]:
result = ffai.generate_response(
    "Write a haiku about coding.",
    prompt_name="haiku",
    temperature=0.9,
    max_tokens=100,
)

print(f"Response:\n{result.response}")

---

## Summary

| Parameter | Where | Purpose |
|-----------|-------|---------|
| `prompt` | top-level positional | The user prompt string |
| `prompt_name` | top-level | Identifies the prompt for templating |
| `history` | top-level | Conversation history (list of message dicts) |
| `options` | top-level | `ResponseOptions` dataclass |
| `**kwargs` | top-level | Provider-specific: `temperature`, `max_tokens`, `tools`, etc. |

**`ResponseOptions` fields:**

| Field | Type | Purpose |
|-------|------|---------|
| `response_model` | `Type[BaseModel]` | Pydantic model for structured output |
| `condition` | `Any` | Skip LLM call if falsy |
| `response_format` | `dict` | Raw JSON mode (`{"type": "json_object"}`) |
| `strict` | `bool` | Strict schema validation |
| `retry_count` | `int` | Max retries on validation failure |
| `retry_prompt` | `str` | Custom retry instructions |
| `history` | `list` | Fallback history (top-level `history` takes precedence) |
| `system_prompt` | `str` | Override system prompt |
| `parsed` | `Any` | Pre-seed the parsed result |